# Structured output with a Pokédex

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

# Load GROQ_API_KEY from the local .env file.
load_dotenv()

groq_model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)


## 1. Pydantic: one Pokémon

In [2]:
class Pokemon(BaseModel):
    """One Pokémon entry returned by the model."""

    name: str = Field(description="The Pokémon name.")
    type: str = Field(description="Its main Pokémon type.")
    level: int = Field(description="A sensible level for this example.")
    evolutions: list[str] = Field(description="Its possible evolution names.")


# This wrapped model returns a Pokemon object instead of free-form text.
pokemon_model = groq_model.with_structured_output(Pokemon)


In [3]:
pokemon = pokemon_model.invoke("Give Pokédex details for Pikachu.")

print(pokemon)
print("Name:", pokemon.name)
print("Evolutions:", pokemon.evolutions)


name='Pikachu' type='Electric' level=25 evolutions=['Pichu', 'Raichu']
Name: Pikachu
Evolutions: ['Pichu', 'Raichu']


## 2. Nested output: a Pokédex list

In [8]:
class PokemonList(BaseModel):
    """A small Pokédex result containing several Pokémon."""

    pokemons: list[Pokemon] = Field(
        description="The Pokémon included in this Pokédex result."
    )


pokedex_model = groq_model.with_structured_output(PokemonList)


In [9]:
pokedex = pokedex_model.invoke(
    "Create a short Pokédex for three of Ash's Kanto Pokémon."
)

for pokemon in pokedex.pokemons:
    print(f"{pokemon.name}: {pokemon.type}")
    print("Evolutions:", pokemon.evolutions)


Pikachu: Electric
Evolutions: ['Raichu']
Bulbasaur: Grass/Poison
Evolutions: ['Ivysaur', 'Venusaur']
Charmander: Fire
Evolutions: ['Charmeleon', 'Charizard']


## 3. TypedDict: a lighter shape

In [10]:
class PokemonDict(TypedDict):
    """One simple Pokémon dictionary."""

    name: str
    type: str
    level: int
    evolutions: list[str]


# This wrapped model returns a regular dictionary.
typed_dict_model = groq_model.with_structured_output(PokemonDict)


In [11]:
pokemon_dict = typed_dict_model.invoke("Give Pokédex details for Eevee.")

print("Name:", pokemon_dict["name"])
print("Evolutions:", pokemon_dict["evolutions"])


Name: Eevee
Evolutions: ['Vaporeon', 'Jolteon', 'Flareon', 'Espeon', 'Umbreon', 'Leafeon', 'Glaceon', 'Sylveon']
